In [ ]:
# Load data
import pandas as pd
url = "https://raw.githubusercontent.com/AlessandroGianfelici/danish_reviews_dataset/refs/heads/master/raw_data.txt"
df = pd.read_csv(url)

# Split to only negative
df_lowrating = df[(df['review_stars'] == 1) | (df['review_stars'] == 2)]

# Sample 100 lines randomly
df_sampled = df_lowrating.sample(n=100, random_state=42)

# Save to csv
output_file = 'sampled_low_rating_reviews.csv'
df_sampled.to_csv(output_file, index=False)

Categories utilized for labeling complaints on the sampled reviews:
1. **Credibility** (Troværdighed): Fraud, ethical issues, and aspects that could be discussed in court.
2. **Product quality** (Produktkvalitet): Overall quality of the core business product/service experienced.
3. **Customer service** (Kundeservice): Service provided in person, by phone, or on the Internet to help ensure a successful flow of the business.
4. **Pricing** (Prissætning): Value compared to competitors, transparency of pricing, and price computation.
5. **Delivery issues** (Leveringsproblemer): Correctness of delivery like timing, correct product/service, and other delivery agreements.
6. **Return/refund issues** (Retur og refundering): Functioning of the return and refund processes.

When a review involved more than one category, the above order was used as a priority list.

The file **sampled_low_rating_reviews.csv** was converted to an Excel file for labeling, and the resulting dataset was converted back to a csv file: **sampled_low_rating_reviews_labeled.csv**.


In [ ]:
# Read labeled dataset
labeled_file = 'sampled_low_rating_reviews_labeled.csv'
df_labeled = pd.read_csv(labeled_file)
print(df_labeled.shape)
df_labeled.head()

# # Manually labeled dataset
# labels = []
# for text in df_lowrating['review_text'].head(100):
#     print(f"\nReview: {text}")
#     label = input("Enter category (Pricing, Production quality, Delivery issue, Packing problem, Customer service, Technical problems, Return/refund issues): ")
#     labels.append(label)

# print("Labeling done")

(100, 5)


,company_name,review_title,review_text,review_stars,label
0,7669662866051230000,Har tidligere været nogenlunde tilfreds…,Har tidligere været nogenlunde tilfreds medlem...,1,Produktkvalitet
1,8202522407772090000,Dårligste dækning,Deres internet er stort set ubrugelig på Frede...,1,Kundeservice
2,-4820715024077190000,Forsendelse virker OK,Forsendelse virker OK. Men er alt for dyr i fo...,1,Prissætning
3,221425046850680000,Fup hjemmeside,Hjemmesiden er fup. Du får aldrig den vare du ...,1,Troværdighed
4,-8509529149368170000,Pas på det forlanger afgift når i…,Pas på det forlanger afgift når i handler,1,Prissætning


In [ ]:
# Sample of low reviews only
import pandas as pd
df = pd.read_excel('sampled_low_rating_reviews_labeled.xlsx')

df_labels = df['label']


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [ ]:
# Train/validation split (75/25)
# Shuffle dataset of only labels
df_shuffled = df_labels.sample(frac=1, random_state=42).reset_index(drop=True)

# Define split
split_index = int(len(df_shuffled) * 0.75)

# Split
train = df_shuffled.iloc[:split_index]
test = df_shuffled.iloc[split_index:]

### Evaluation function

In [ ]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report,
    precision_recall_fscore_support
)

def evaluate_model(y_true, y_pred):
    confusion_table = pd.crosstab(y_true, y_pred, rownames=['Actual'], colnames=['Predicted'])
    print("\n--- Confusion Table ---")
    print(confusion_table)

    # Global Metrics
    # 'macro' calculates the mean of metrics for each label
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
    accuracy = accuracy_score(y_true, y_pred)

    print(f"\n--- Global Performance ---")
    print(f"Total Accuracy: {accuracy:.2%}")
    print(f"Macro Precision: {precision:.2%}")
    print(f"Macro Recall: {recall:.2%}")
    print(f"Macro F1-score: {f1:.2%}")

    print("\n--- Detailed Metrics per Category ---")
    print(classification_report(y_true, y_pred))

    return precision, recall, f1, accuracy

## 1. Zero-Shot Baseline: Evaluate zero-shot classification

### 1.1 Using: joeddav/xlm-roberta-large-xnli

In [ ]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli")

# Help the model to understand the context better in Danish.
candidate_labels = [
    "Troværdighed: Svindel, fup, etiske problemer eller forhold der kan ende i retten.",
    "Produktkvalitet: Den generelle oplevelse af kvaliteten af produktet eller kernerydelsen.",
    "Kundeservice: Service ydet personligt, via telefon eller internet for at sikre et godt forløb.",
    "Prissætning: Værdi ift. konkurrenter, gennemsigtighed i priser og prisberegning.",
    "Leveringsproblemer: Korrekt levering ift. tid, det rette produkt og andre leveringsaftaler.",
    "Retur og refundering: Hvordan processen for returnering og tilbagebetaling fungerer."
]
hypothesis_template = "Denne anmeldelse beskriver et problem med {}"

# Join 3 columns info in one
df_labeled['model_input'] = df_labeled.apply(lambda x:
    f"Titel: {x['review_title']}. "
    f"Anmeldelse: {x['review_text']}. "
    f"Kunden gav {x['review_stars']} stjerner.", axis=1)

def classify_review(text):
    result = classifier(text, candidate_labels, hypothesis_template=hypothesis_template)
    # Return the label with the highest score
    return result['labels'][0]

# Predict
df_labeled['nli_label'] = df_labeled['model_input'].apply(classify_review)
# Keep in label only the category name (before ':')
df_labeled['nli_label'] = df_labeled['nli_label'].str.split(":").str[0].str.strip()


# ### EVALUATE ###
nli_precision, nli_recall, nli_f1_score, nli_accuracy = evaluate_model(df_labeled['label'], df_labeled['nli_label'])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Device set to use cpu



--- Confusion Table ---
Predicted             Kundeservice  Leveringsproblemer  Prissætning  \
Actual                                                                
Kundeservice                    18                   0            0   
Leveringsproblemer               3                   1            0   
Prissætning                      2                   1            1   
Produktkvalitet                 12                   0            0   
Retur og refundering             1                   0            0   
Troværdighed                    10                   0            1   

Predicted             Produktkvalitet  Retur og refundering  Troværdighed  
Actual                                                                     
Kundeservice                        2                     2             3  
Leveringsproblemer                  2                     1             0  
Prissætning                         3                     0             0  
Produktkvalitet           

ValueError: Found input variables with inconsistent numbers of samples: [100, 44085]

In [ ]:
# NLI model: Facebook's BART-Large-MNLI
from transformers import pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Select random review
customer_review = df_lowrating['review_text'].sample(1).iloc[0]
categories = ['Troværdighed', 'Produkt kvalitet', 'Kundeservice', 'Prissætning', 'Leveringsproblemer', 'Return og refundering']

result = classifier(customer_review, categories)
print(result)

Device set to use cuda:0


{'sequence': 'Den rene elendige oplevelse. \nJeg bestilte sushi gennem goodtogo, førpris 299 kr. nupris 99 kr. For st støtte et godt koncept igennem dem. \n\nForan mig lå 12 stk indtørrede sushi med frossen tangsalat. En ting er at det er et tilbud fra deres side mellem 20:30-21.00 hvor de skal af med overskud men det hænger ikke sammen med førprisen og deres såkaldte kvalitet. Aldrig har jeg fået så kedeligt sushi.\nÆrgerligt at goodtogo vil ligge navn til sådan noget.\n\nTænk at skulle betale fuld pris.', 'labels': ['Pricing', 'Product quality', 'Delivery issues', 'Customer service', 'Credibility', 'Return/refund issues'], 'scores': [0.2594008147716522, 0.2459402084350586, 0.14722475409507751, 0.12891696393489838, 0.11894106864929199, 0.0995762050151825]}


### 1.2 Using: "google-bert/bert-base-multilingual-cased"

In [ ]:
# 2. LLM: zero shot
# Select random review and define categories
!pip install transformers==4.46.2 accelerate==1.1.1 bitsandbytes==0.45.2
customer_review = df_lowrating['review_text'].sample(1).iloc[0]
categories = ['Troværdighed', 'Produkt kvalitet', 'Kundeservice', 'Prissætning', 'Leveringsproblemer', 'Return og refundering']

import torch
from transformers import pipeline, BitsAndBytesConfig

# 1. Setup the "Compression" config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

# 2. Initialize the pipeline with the config
pipe = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.1-8B-Instruct",
    token="YOUR_TOKEN",
    model_kwargs={
        "quantization_config": quantization_config,
        "device_map": "auto"
    }
)
messages = [
    {
        "role": "system",
        "content": f"You are a classification assistant. Categorize the user's message into exactly one of these categories: {', '.join(categories)}. Respond with only the category name."
    },
    {
        "role": "user",
        "content": customer_review
    },
]
outputs = pipe(messages, max_new_tokens=20) # limited tokens to avoid rambling
print(outputs[0]["generated_text"][-1]["content"])
text = outputs[0]["generated_text"]

user_message = text[-2]["content"]
assistant_label = text[-1]["content"]


print(f"Customer review: {user_message}")
print(f"Category: {assistant_label}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Kundeservice
Customer review: Kære Politi.dk 
Midt i nat kl. 02.53 ringer i 3 gange fordi i har fået en melding om at en Anton holder fest i København. Jeg hedder ikke Anton, bor i Aarhus, og er træt nok i forvejen, fordi vi er skiftet fra vinter til sommertid. Goddamn en mandag jeg har haft nu. 

Mvh. En døende træt herre.
Category: Kundeservice


In [ ]:
# 2. LLM: few shots
# Select random review and define categories
!pip install transformers==4.46.2 accelerate==1.1.1 bitsandbytes==0.45.2
customer_review = df_lowrating['review_text'].sample(1).iloc[0]
categories = ['Troværdighed', 'Produkt kvalitet', 'Kundeservice', 'Prissætning', 'Leveringsproblemer', 'Return og refundering']

import torch
from transformers import pipeline, BitsAndBytesConfig

# 1. Setup the "Compression" config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

# 2. Initialize the pipeline with the config
pipe = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.1-8B-Instruct",
    token="YOUR TOKEN",
    model_kwargs={
        "quantization_config": quantization_config,
        "device_map": "auto"
    }
)

few_shot_guide = [
    {"role": "user", "content": "Leveringen tog alt for lang tid om at komme."},
    {"role": "assistant", "content": "Leveringsproblemer"},

    {"role": "user", "content": "Scam! Hold jee væk!"},
    {"role": "assistant", "content": "Troværdighed"},

    {"role": "user", "content": "Falske varer"},
    {"role": "assistant", "content": "Produkt kvalitet"},


]

messages = [
    {
        "role": "system",
        "content": f"You are a classification assistant. Categorize into one of: {', '.join(categories)}. Respond with only the category name."
    },

    *few_shot_guide,

    {
        "role": "user",
        "content": customer_review
    },
]

outputs = pipe(messages, max_new_tokens=20) # limited tokens to avoid rambling
print(outputs[0]["generated_text"][-1]["content"])
text = outputs[0]["generated_text"]

user_message = text[-2]["content"]
assistant_label = text[-1]["content"]


print(f"Customer review: {user_message}")
print(f"Category: {assistant_label}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Kundeservice
Customer review: Har aldrig nogen sinde oplevet at en kundeservice hentyder til man lyver.
Ringede ind i dag da jeg ikke har noget netværk når jeg er hjemme og på arbejdet.
Medarbejderen siger inddirekte at jeg lyver, og det simpelthen ikke kan passe.

Nu skal jeg så tage en masse test for at bevise over for jer at jeg ikke har noget net.
Har aldrig oplevet noget så virkelig ubehøvlet som dette.

Jeg forventer at der kommer styr på dette og i finder en løsning. for jeg er langt fra tilfreds med det her.
Category: Kundeservice


### 1.2.1 - Zero-shot prompts

### 1.2.2 Few-shot propts (3-5 examples)

## 2. Fine-tune on 2 pre-trained transformer models

In [ ]:
# 1. Multilingual BERT - use for testing, add training arg
from transformers import pipeline
pipe = pipeline("fill-mask", model="google-bert/bert-base-multilingual-cased")

customer_review = df['review_text'].sample(1).iloc[0]

# Template to extract the 'problem'
template = f"{customer_review} The main problem in this experience was the [MASK]."
results = pipe(template)

print(f"Sentence: {customer_review}\n")
for res in results:
    print(f"Predicted Issue: {res['token_str']} ({res['score']:.2%})")

Some weights of the model checkpoint at google-bert/bert-base-multilingual-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


Sentence: De trak pengene fra min konto 10.02.2021
Men de har endnu ikke leveret den bestilte hundesele med navn 😡
Hold jer fra dette fup firma 😡🤬😡
Lisbeth Vad Denmark

Predicted Issue: accident (0.42%)
Predicted Issue: loss (0.40%)
Predicted Issue: death (0.38%)
Predicted Issue: war (0.37%)
Predicted Issue: fact (0.37%)


In [ ]:
# 2. Danish BERT
from transformers import pipeline

pipe = pipeline("fill-mask", model="Maltehb/danish-bert-botxo")
customer_review = df['review_text'].sample(1).iloc[0]

# Template to extract the 'problem'
template = f"{customer_review} The main problem in this experience was the [MASK]."
results = pipe(template)

print(f"Sentence: {customer_review}\n")
for res in results:
    print(f"Predicted Issue: {res['token_str']} ({res['score']:.2%})")

Device set to use cuda:0


Sentence: Jeg er desværre også blevet synet og bedraget af dette firma. Her troede jeg at jeg havde fundet et godt tilbud. NOT. Fup og fidus. Trækker pengene fra ens konto uden at sende den bestilte vare. Når man så kontakter firmaet, bliver man bare spidset af med at der er forsinkelse på forsinkelse . Hold jer væk fra dette fup firma. Det bliver kun en kamp at få sine penge eller den bestile vare.

Predicted Issue: rest (23.18%)
Predicted Issue: points (13.56%)
Predicted Issue: city (8.55%)
Predicted Issue: ##und (4.32%)
Predicted Issue: virus (2.32%)
